# SteppeDNA: ESM-2 650M Embeddings (Upgrade from 8M)

This notebook generates protein embeddings using the larger ESM-2 650M model
(esm2_t33_650M_UR50D) instead of the current 8M model (esm2_t6_8M_UR50D).

The 650M model has significantly better protein understanding and should
improve variant discrimination, especially for non-BRCA2 genes.

**Requirements:** Google Colab with GPU (T4 sufficient, A100 faster)

**Output:** `esm2_650m_embeddings.pkl` - drop-in replacement for current embeddings

In [ ]:
!pip install -q torch transformers biopython pandas numpy scikit-learn

In [ ]:
from google.colab import files
print("Upload master_training_dataset.csv:")
uploaded = files.upload()

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, EsmModel
from sklearn.decomposition import PCA
import pickle
import requests

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load ESM-2 650M (vs current 8M)
MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'

print('Loading ESM-2 650M...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {total_params/1e6:.0f}M parameters')

In [ ]:
# Fetch reference sequences from UniProt
UNIPROT_IDS = {
    'BRCA1': 'P38398', 'BRCA2': 'P51587', 'PALB2': 'Q86YC2',
    'RAD51C': 'O43502', 'RAD51D': 'O75771',
}
GENE_AA_LENGTHS = {'BRCA1': 1863, 'BRCA2': 3418, 'PALB2': 1186, 'RAD51C': 376, 'RAD51D': 328}
GENE_SEQUENCES = {}

for gene, uid in UNIPROT_IDS.items():
    resp = requests.get(f'https://rest.uniprot.org/uniprotkb/{uid}.fasta')
    if resp.ok:
        seq = ''.join(resp.text.strip().split('\n')[1:])
        GENE_SEQUENCES[gene] = seq
        print(f'{gene}: {len(seq)} AA')

print(f'Fetched {len(GENE_SEQUENCES)}/5 sequences')

In [ ]:
# Load dataset and generate per-variant embeddings
df = pd.read_csv('master_training_dataset.csv')
print(f'Dataset: {len(df)} variants')

# Detect the gene column name (could be 'gene' or 'gene_name')
gene_col = None
for col in ['gene_name', 'gene', 'Gene', 'Gene_Name']:
    if col in df.columns:
        gene_col = col
        break

if gene_col:
    print(f'Gene column: "{gene_col}"')
    print(f'Gene distribution:\n{df[gene_col].value_counts().to_string()}')
else:
    print('WARNING: No gene column found! Will treat all as BRCA2.')

CONTEXT_WINDOW = 50
BATCH_SIZE = 16

all_embeddings = {}

for gene in ['BRCA1', 'BRCA2', 'PALB2', 'RAD51C', 'RAD51D']:
    if gene_col:
        gene_df = df[df[gene_col] == gene]
    else:
        gene_df = df if gene == 'BRCA2' else pd.DataFrame()
    
    if len(gene_df) == 0:
        print(f'{gene}: 0 variants, skipping')
        continue
    
    full_seq = GENE_SEQUENCES.get(gene)
    if not full_seq:
        print(f'{gene}: No sequence, skipping')
        continue
    
    sequences = []
    indices = []
    
    for idx, row in gene_df.iterrows():
        # Try multiple column names for AA position
        aa_pos = None
        if 'relative_aa_pos' in row.index and pd.notna(row['relative_aa_pos']):
            aa_pos = int(row['relative_aa_pos'] * GENE_AA_LENGTHS[gene])
        elif 'AA_pos' in row.index and pd.notna(row['AA_pos']):
            aa_pos = int(row['AA_pos'])
        elif 'cDNA_pos' in row.index and pd.notna(row['cDNA_pos']):
            aa_pos = int(row['cDNA_pos']) // 3
        else:
            aa_pos = len(full_seq) // 2
        
        aa_pos = max(0, min(aa_pos, len(full_seq) - 1))
        start = max(0, aa_pos - CONTEXT_WINDOW)
        end = min(len(full_seq), aa_pos + CONTEXT_WINDOW + 1)
        sequences.append(full_seq[start:end])
        indices.append(idx)
    
    print(f'\n{gene}: Generating {len(sequences)} embeddings...')
    
    for i in range(0, len(sequences), BATCH_SIZE):
        batch_seqs = sequences[i:i+BATCH_SIZE]
        batch_idx = indices[i:i+BATCH_SIZE]
        
        inputs = tokenizer(batch_seqs, return_tensors='pt', padding=True,
                          truncation=True, max_length=128)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            hidden = outputs.last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        
        for j, idx in enumerate(batch_idx):
            all_embeddings[idx] = pooled[j].cpu().numpy()
        
        if (i // BATCH_SIZE) % 100 == 0 and i > 0:
            print(f'  {gene}: {i}/{len(sequences)} done')
    
    print(f'  -> {gene}: {len(gene_df)} embeddings done')

print(f'\nTotal embeddings: {len(all_embeddings)}')

In [ ]:
# PCA reduction to 20 components
sorted_indices = sorted(all_embeddings.keys())
embedding_matrix = np.array([all_embeddings[i] for i in sorted_indices])
print(f'Raw: {embedding_matrix.shape}')

pca = PCA(n_components=20, random_state=42)
pca_embeddings = pca.fit_transform(embedding_matrix)
print(f'PCA: {pca_embeddings.shape}')
print(f'Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Also compute cosine similarity and L2 shift (like current pipeline)
cosine_sims = []
l2_shifts = []
for i, idx in enumerate(sorted_indices):
    emb = embedding_matrix[i]
    mean_emb = embedding_matrix.mean(axis=0)
    cos = np.dot(emb, mean_emb) / (np.linalg.norm(emb) * np.linalg.norm(mean_emb) + 1e-10)
    l2 = np.linalg.norm(emb - mean_emb)
    cosine_sims.append(cos)
    l2_shifts.append(l2)

# Package output
output = {
    'embeddings_pca': {idx: pca_embeddings[j] for j, idx in enumerate(sorted_indices)},
    'cosine_sim': {idx: cosine_sims[j] for j, idx in enumerate(sorted_indices)},
    'l2_shift': {idx: l2_shifts[j] for j, idx in enumerate(sorted_indices)},
    'pca_model': pca,
    'model_name': MODEL_NAME,
    'n_components': 20,
    'variance_explained': float(pca.explained_variance_ratio_.sum()),
    'embedding_dim': embedding_matrix.shape[1],
}

with open('esm2_650m_embeddings.pkl', 'wb') as f:
    pickle.dump(output, f)

print(f'\nSaved esm2_650m_embeddings.pkl ({embedding_matrix.shape[1]}-dim -> 20 PCA)')

In [ ]:
# Download
from google.colab import files
files.download('esm2_650m_embeddings.pkl')

print('\n--- DONE ---')
print('Place esm2_650m_embeddings.pkl in SteppeDNA/data/')
print('Update generate_esm2_embeddings.py to use 650M model for production')
print('Retrain with: python scripts/train_universal_model.py')